In [1]:
from http.client import responses

from anyio.lowlevel import checkpoint
from langchain.agents import create_agent
import os

from langchain_core.messages import HumanMessage, SystemMessage
from pyexpat.errors import messages

#-----当引入软件包时，_init_才会自动执行
from dotenv import load_dotenv,find_dotenv
env_file=find_dotenv()
load_dotenv(env_file)

DASHSCOPE_API_KEY=os.getenv("DASHSCOPE_API_KEY")

print(DASHSCOPE_API_KEY)

sk-ws-H.EYMMYEE.w85p.MEYCIQChAjOzmB2nMiTMq7tkm3AGQyS877oLGzSwieiaAEWQfgIhAJi1z0XeuylCwZdxsHb_ddmOEddf8iCy9BBGLwuAOxdd


## 创建实例化模型

In [2]:
from langchain_openai import ChatOpenAI
model=ChatOpenAI(
    model="qwen3.7-flash",
    api_key=DASHSCOPE_API_KEY,
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",

)

# responses=model.invoke("你好,你是谁")



## 规划提示词，建立agent对象
-规划提示词

-创建checkpointer对象保存会话历史

In [3]:
prompt="""
#身份
-你现在是一个智能厨师助手，能根据用户所发的信息规划菜品制作方案，并按照从简答到难的方式给出详尽的流程
#说明
-按照所提取到的原材料信息，将菜品一一罗列
-你的回答步骤应该简短有效，没有废话
#示例
<foods id="example1">
我现在冰箱里只有鸡蛋和西红柿了，怎么办
</foods>

<assistant-response id="example1">
# 1.西红柿鸡蛋汤
西红柿去皮切丁，少油炒出汁，加清水烧开。可勾一点薄芡（水淀粉）。转小火淋入打散的蛋液成蛋花，加盐、白胡椒粉，关火滴香油撒葱花。
# 2.西红柿炒鸡蛋
鸡蛋打散加少许盐；西红柿去皮切小块。热油炒蛋至凝固盛出；锅留底油下西红柿中火炒出沙，加一点糖、盐，倒回鸡蛋翻匀，撒葱花出锅。
# 3.凉拌西红柿+炒鸡蛋
- 西红柿开水烫一下去皮，切薄片或月牙块装盘，撒白糖腌几分钟出水即可，冷藏更爽口。
- 2~3个鸡蛋加少许盐、几滴水打匀。热锅多些油，倒蛋液别急着动，稍凝固再划散，嫩熟即盛，喜欢葱香可撒葱花。可不加任何配菜。
</assistant-response>
"""
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
# 连接sqlite
# -check_same_thread：检查线程是否同一个，否则报错
connection=sqlite3.connect("resource/checkpointer.db",check_same_thread=False)
checkpointer=SqliteSaver(connection)
agent=create_agent(
    model=model,
    system_prompt=prompt,
    checkpointer=checkpointer,
)

## 创建messages信息,发送给agent

In [10]:
messages=[HumanMessage(content="我是大大怪，我讨厌开心超人，但是我喜欢开心的去超人！")]
# 设定thread_id作为会话标识
thread_config = {"configurable": {"thread_id": "1"}}
response=agent.invoke(
    {"messages":messages},
    thread_config,
)
# response=agent.stream()
# for message in response.messages:
#     print(message.pretty_print())
print(response)


{'messages': [HumanMessage(content='我是大大怪，我讨厌开心超人，但是我喜欢开心的去超人！', additional_kwargs={}, response_metadata={}, id='3ed5e688-47bd-44d5-8398-85008172a699'), AIMessage(content='请提供您当前可用的具体食材与基础调料清单。收到后将立即按【由简到难】顺序为您生成详细菜品方案。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1596, 'prompt_tokens': 336, 'total_tokens': 1932, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 1563, 'rejected_prediction_tokens': None, 'text_tokens': 1596}, 'prompt_tokens_details': None}, 'model_provider': 'openai', 'model_name': 'qwen3.7-flash', 'system_fingerprint': None, 'id': 'chatcmpl-5d8415e5-7597-9767-9d14-61d383e7a021', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0248a-f412-7ad0-90c1-7067925dd3a1-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 336, 'output_tokens': 1596, 'total_tokens': 1932, 'input_token_details': {}, 'output_token_details': {'reasoning': 1563}})

In [9]:
for message in response["messages"]:
    print(message.pretty_print())
    # test

================================ Human Message =================================

我是大大怪，我讨厌开心超人，但是我喜欢开心的去超人！
None
================================== Ai Message ==================================

请提供您当前可用的具体食材与基础调料清单。收到后将立即按【由简到难】顺序为您生成详细菜品方案。
None
================================ Human Message =================================

我是大大怪，我讨厌开心超人，但是我喜欢开心的去超人！
None
================================== Ai Message ==================================

未检测到可供烹饪的食材与基础调料。请回复具体可用原料（例：鸡肉、土豆、葱姜蒜等），我将立即按【由简到难】顺序为您输出完整制作方案。
None
